## vLLM Serving Demo

A minimal, self-contained demo showing how to deploy and serve a local LLM using `vLLM` — from offline inference to a running OpenAI-compatible API server, on a single GPU.

### What this demonstrates

- Installing and running vLLM in a resource-constrained environment (single T4 GPU, Google Colab)
- Offline batch inference with vLLM's `LLM` class
- Deploying a local model as an OpenAI-compatible API server (`vllm serve`)
- Continuous batching: handling multiple concurrent requests efficiently
- Token streaming, matching the UX of hosted chat APIs
- Practical constraints and trade-offs of running small open-weight models on limited hardware

#### 1. Check GPU

In [2]:
!nvidia-smi

Thu Sep 10 15:44:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

#### 2. Install vLLM

In [3]:
!pip install vllm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.7 MB/s eta 0:00:00


#### 3. Remove torchaudio (avoids CUDA version mismatch with torch)

In [4]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0


NOTE: **→ Runtime → Restart session (manual step, not code)**

#### 4. First cell after restart: workaround + import

In [13]:
# Prevent transformers from trying to import torchaudio (not needed for text-only LLMs,
# and Colab's default torch/torchaudio CUDA versions often don't match)
import transformers.utils.import_utils as iu
iu.is_torchaudio_available = lambda: False

from vllm import LLM, SamplingParams
print("Import OK")

Import OK


#### 5. Write an offline inference test script to a file

In [14]:
%%writefile test_vllm.py
import transformers.utils.import_utils as iu
iu.is_torchaudio_available = lambda: False

from vllm import LLM, SamplingParams

llm = LLM(model="Qwen/Qwen2.5-1.5B-Instruct")
params = SamplingParams(temperature=0.3, max_tokens=150)

messages = [
    {"role": "user", "content": "Explain in two sentences what photosynthesis is."}
]

output = llm.chat(messages, params)
print(output[0].outputs[0].text)

Overwriting test_vllm.py


#### 6. Run it with the correct interpreter

In [15]:
import sys
print(sys.executable)

/usr/bin/python3


In [18]:
!{sys.executable} test_vllm.py

INFO 09-10 16:44:25 [api_utils.py:286] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
INFO 09-10 16:44:26 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-10 16:44:26 [model.py:2302] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 09-10 16:44:26 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-10 16:44:26 [model.py:2021] Using max model len 32768
INFO 09-10 16:44:26 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-10 16:44:27 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=22385) INFO 09-10 16:44:32 [core.py:123] Initializing a V1 LLM engine (v0.29.0) with config: model='Qwen/Qwen2.5-1.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', skip_tokenizer_

#### 7. Start the OpenAI-compatible server in the background

In [19]:
import subprocess

server_process = subprocess.Popen(
    [
        "vllm", "serve", "Qwen/Qwen2.5-1.5B-Instruct",
        "--port", "8000",
        "--gpu-memory-utilization", "0.7"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
print("Server starting in background, PID:", server_process.pid)

Server starting in background, PID: 23382


#### 8. Wait until the server is ready

In [20]:
import time
import requests

def wait_for_server(url="http://localhost:8000/health", timeout=180):
    start = time.time()
    while time.time() - start < timeout:
        try:
            r = requests.get(url)
            if r.status_code == 200:
                print("Server ready!")
                return True
        except requests.exceptions.ConnectionError:
            pass
        time.sleep(3)
        print("...waiting for server")
    raise TimeoutError("Server did not respond within the expected time")

wait_for_server()

...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
...waiting for server
Server ready!


True

#### 9. Install OpenAI client (if needed) and send a single request

In [21]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")

response = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[{"role": "user", "content": "Hi, who are you?"}]
)
print(response.choices[0].message.content)

I'm an AI assistant created by Alibaba Cloud, and I'm here to help with your questions. How may I assist you today?


#### 10. Concurrent requests (demonstrates continuous batching)

In [22]:
import time
from concurrent.futures import ThreadPoolExecutor

prompts = [
    "What is the capital of Poland?",
    "Name three planets in the Solar System.",
    "What is an algorithm?",
    "Write one sentence about coffee.",
]

def ask(prompt):
    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct",
        messages=[{"role": "user", "content": prompt}]
    )
    return r.choices[0].message.content

start = time.time()
with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(ask, prompts))
elapsed = time.time() - start

for p, r in zip(prompts, results):
    print(f"Q: {p}\nA: {r}\n---")

print(f"Time for {len(prompts)} concurrent requests: {elapsed:.2f}s")

Q: What is the capital of Poland?
A: The capital of Poland is Warsaw.
---
Q: Name three planets in the Solar System.
A: Three planets in our Solar System are Mercury, Venus, and Earth.
---
Q: What is an algorithm?
A: An algorithm is a well-defined procedure or set of rules that specifies how to solve a problem or perform a task. It's like a recipe for solving a puzzle or completing a task.

Here are some key points about algorithms:

1. Definition: An algorithm is a finite sequence of instructions or steps designed to accomplish a specific goal or solve a particular problem.

2. Purpose: Algorithms are used in various fields such as computer science, mathematics, engineering, and everyday life.

3. Steps: They consist of a series of logical steps that must be followed in order to arrive at the desired outcome.

4. Efficiency: The efficiency of an algorithm refers to its time complexity (how long it takes to run) and space complexity (how much memory it uses).

5. Examples:
   - Sorting

#### 11. Streaming response

In [23]:
stream = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[{"role": "user", "content": "Tell a short story about a cat."}],
    stream=True
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

Once upon a time, there was a little kitten named Whiskers who lived in a cozy house with her human family. She loved to explore and play outside, but she always came back home safe and sound.

One day, while playing with her toys in the backyard, Whiskers discovered an old abandoned mouse trap. Curious and adventurous, she decided to investigate further. As she sniffed around, she found some interesting things inside - bits of cheese, a small hole for escape, and even a tiny spider!

Excited by her discovery, Whiskers started nibbling on the cheese. But as soon as she did, the trap sprung! A loud hissing noise filled the air, and Whiskers jumped up in fright. Her paws were scratched, and she was covered in sticky glue.

But instead of being scared or upset, Whiskers realized that she had stumbled upon something very special. The trap was actually designed to catch mice like herself! And so, she became fascinated by the science behind it all.

From that day forward, Whiskers spent ever

#### 12. Cleanup

In [24]:
server_process.terminate()
server_process.wait()
print("Server stopped.")

Server stopped.
